# Week 1 · Hand-Written Multi-Head Attention

> **Learning mode**: you implement the core algorithm yourself. This notebook only provides scaffolding, shape hints, and reference checks for verification.

## Goals For This Week

1. **Implement single-head attention from scratch** to understand what Q/K/V are doing as content-based addressing.
2. **Implement multi-head attention from scratch** to understand the equivalence `MHA(x) = Concat(heads)·W_O`.
3. **Prove the equivalence** between concatenating multiple heads and slicing `W_Q, W_K, W_V` into `h` blocks and computing them separately.
4. **Add causal masking** for decoder-style self-attention.
5. **Compare numerically with `nn.MultiheadAttention`** to verify that your implementation is correct.

## Deliverables

- `mha_scratch.py` extracted from this notebook and rewritten again without looking at the reference.
- A one-page derivation note covering shape flow and the equivalence proof in `../notes/week1_reviewer.md`.

## Reviewer Questions To Ask Yourself

- Why do we divide by `sqrt(d_k)`? What goes wrong if we do not?
- Why is softmax applied along the key dimension? What would happen along the query dimension instead?
- What is the difference between multi-head attention and a single wider head? Think in terms of expressivity vs. parameterization.
- Should the causal mask be applied before or after softmax, and why?


## 0. Environment and Tools


In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(42)
print("device:", device, "| torch:", torch.__version__)


## 1. Single-Head Attention (You Implement It)

**Input**: `x` of shape `(B, T, d_model)`

**What you need to do**:
1. Project `x` into `Q, K, V` using three linear layers, each with shape `(B, T, d_k)`.
2. Compute `scores = Q @ K^T / sqrt(d_k)` with shape `(B, T, T)`.
3. Optionally apply a causal mask.
4. Compute `attn = softmax(scores, dim=?)` and think carefully about which dimension softmax should use.
5. Compute `out = attn @ V` with shape `(B, T, d_k)`.

**Hints**:
- `torch.tril(torch.ones(T, T))` can be used to build a lower-triangular mask.
- A standard masking pattern is `scores.masked_fill(mask == 0, float('-inf'))` **before softmax**.
- For initialization, use `nn.Linear(d_model, d_k, bias=False)` to reduce moving parts at first.


In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int, d_k: int, causal: bool = True):
        super().__init__()
        # TODO: define W_q, W_k, W_v as three linear layers
        # TODO: store d_k and causal
        raise NotImplementedError("your implementation")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        # TODO:
        # 1. project Q, K, V
        # 2. scores = Q @ K^T / sqrt(d_k)
        # 3. causal mask (if self.causal)
        # 4. softmax -> attention weights
        # 5. out = attn @ V
        # return out, attn  (return attn too for visualization)
        raise NotImplementedError("your implementation")


In [ ]:
# Small test: forward pass works and shapes are correct
B, T, d_model, d_k = 2, 5, 16, 8
x = torch.randn(B, T, d_model)
sha = SingleHeadAttention(d_model, d_k, causal=True)
out, attn = sha(x)
assert out.shape == (B, T, d_k), f"got {out.shape}"
assert attn.shape == (B, T, T)
# Causal check: attention should be lower triangular
assert torch.allclose(attn.triu(1), torch.zeros_like(attn), atol=1e-6), "causal mask did not work"
# Each row should sum to 1
assert torch.allclose(attn.sum(-1), torch.ones(B, T), atol=1e-5)
print("single-head basic checks passed")


## 2. Multi-Head Attention (You Implement It)

**Core equivalence**: concatenating `h` independent heads is equivalent to treating `W_Q` as one large projection matrix, reshaping the output into `(B, T, h, d_k)`, permuting it into `(B, h, T, d_k)`, computing attention per head, and then concatenating back.

**What you need to do**:
1. Use a single projection for Q/K/V to produce tensors of shape `(B, T, d_model)`, where `d_model = h * d_k`.
2. Reshape and transpose into `(B, h, T, d_k)`.
3. Compute attention independently for each head using batched matrix multiplication.
4. Concatenate from `(B, h, T, d_k)` back to `(B, T, h*d_k)`.
5. Apply the output projection `W_O` from `(B, T, d_model)` to `(B, T, d_model)`.

**Key APIs**:
- `.view()` / `.reshape()` for shape changes
- `.transpose(1, 2)` to swap the head and sequence dimensions
- `torch.matmul` supports broadcasting, so `(B, h, T, d_k) @ (B, h, d_k, T)` directly gives the attention scores for all heads

**Reviewer questions**:
- Why does reshape + transpose allow all heads to be computed with a single matrix multiplication pipeline?
- What is the difference between `.view()` and `.reshape()`? When will `.view()` fail?


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, causal: bool = True):
        super().__init__()
        assert d_model % num_heads == 0
        # TODO: store d_model, num_heads, d_k = d_model // num_heads, and causal
        # TODO: define W_q, W_k, W_v, W_o (all are d_model -> d_model)
        raise NotImplementedError("your implementation")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        B, T, _ = x.shape
        # TODO:
        # 1. project Q, K, V: (B, T, d_model)
        # 2. reshape to (B, T, h, d_k) then transpose to (B, h, T, d_k)
        # 3. scores = Q @ K.transpose(-2, -1) / sqrt(d_k) -> (B, h, T, T)
        # 4. causal mask
        # 5. softmax -> attn
        # 6. out = attn @ V -> (B, h, T, d_k)
        # 7. transpose + reshape back to (B, T, d_model)
        # 8. apply W_o
        # return out, attn
        raise NotImplementedError("your implementation")


In [ ]:
# Shape test
B, T, d_model, h = 2, 7, 32, 4
x = torch.randn(B, T, d_model)
mha = MultiHeadAttention(d_model, h, causal=True)
out, attn = mha(x)
assert out.shape == (B, T, d_model)
assert attn.shape == (B, h, T, T)
print("MHA shapes are correct")


## 3. Numerical Comparison With `nn.MultiheadAttention`

**Goal**: verify that your implementation is numerically equivalent up to small floating-point differences.

**Gotcha**: PyTorch's `nn.MultiheadAttention` defaults to `batch_first=False`, so you should pass `batch_first=True`. You also need to manually copy weights from your module into PyTorch's module, or the other way around.

This section is intentionally left for you to wire up yourself. The key skill here is **reading PyTorch docs and aligning parameter shapes correctly across two implementations**. If you get stuck, ask.


In [ ]:
# TODO: construct torch.nn.MultiheadAttention, copy your W_q/W_k/W_v/W_o weights,
# feed the same input through both implementations, and verify output error < 1e-5


## 4. Equivalence Proof (Write It In `notes/week1_reviewer.md`)

**Claim**: using `h` independent heads, each with its own `W_q^{(i)}, W_k^{(i)}, W_v^{(i)} ∈ R^{d_model × d_k}`, then concatenating their outputs and applying `W_o`, is equivalent to the reshape-based formulation.

Your write-up should include:

1. The mathematical definition of both forms.
2. A proof that a block-diagonal-style view of `W_q` behaves the same under "project + reshape + per-head computation" as "each head projects independently".
3. A discussion of parameter count: how `4 · d_model²` for Q/K/V/O compares with a single-head setup using `d = d_model`.

**Why this matters**: this is one of the key ideas behind why MHA is both efficient and expressive. Later variants like Flash Attention, MQA, and GQA are easier to reason about once this equivalence is clear.


## 5. Next Steps

- [ ] Finish all core cells in this notebook and pass the tests.
- [ ] Close the notebook and rewrite the implementation from scratch in `mha_scratch.py`.
- [ ] Write `../notes/week1_reviewer.md`, including the equivalence proof and your answers to the reviewer questions.
- [ ] Re-read Figure 2 of *Attention Is All You Need* using your own implementation as the reference point.
- [ ] Read *Formal Algorithms for Transformers* by Phuong & Hutter.
